# CHATBOT CIMA — Actividad 6
## Evaluación, validación y ajuste de decisiones

Este notebook consolida en una sola secuencia el proceso de evaluación de los modelos de la Actividad 5. El código ejecutable vive en `fuentes/evaluacion.py`; aquí se prepara el entorno, se validan los insumos, se ejecuta el análisis, se revisan las salidas y se exportan los resultados.

**Flujo de trabajo:** preparar entorno → verificar datos y modelos → evaluar métricas y matrices → calcular ROC/AUC y validación cruzada → seleccionar umbrales con predicciones *out-of-fold* (OOF) → comparar modelos y baseline → simular políticas operativas → validar y exportar resultados.

> **Alcance:** el dataset es sintético. La prueba reservada de la Actividad 5 contiene 26 registros, incluidos solo dos casos `AMBIGUA`. Los resultados son exploratorios; no autorizan decisiones financieras automáticas ni despliegue productivo.

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alfredosalazar-ai/deep-learning-experiments/blob/main/Actividad6/fuentes/CIMA_Actividad6_Evaluacion_Validacion.ipynb)

## 0. Preparar el entorno y localizar el proyecto

Esta celda utiliza el repositorio público `deep-learning-experiments` cuando `Actividad6/` ya está disponible. Si no encuentra los archivos requeridos, permite cargar el ZIP de preparación `CIMA_Actividad6_GitHub.zip`.

No se modifican ni se sobrescriben los modelos originales de la Actividad 5. Tampoco se ejecuta un `git push` automático: la publicación de resultados en GitHub se realiza manualmente después de revisarlos.

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

REPO_URL = "https://github.com/alfredosalazar-ai/deep-learning-experiments.git"
REPO_DIR = Path("/content/deep-learning-experiments")


def es_proyecto_actividad6(ruta: Path) -> bool:
    """Comprueba que una carpeta contiene los insumos necesarios de A6."""
    requeridos = [
        ruta / "requirements.txt",
        ruta / "fuentes" / "evaluacion.py",
        ruta / "modelos" / "Regresion_Logistica.joblib",
        ruta / "modelos" / "LinearSVC.joblib",
        ruta / "datos" / "datos_limp" / "entrenamiento.csv",
        ruta / "datos" / "datos_limp" / "prueba.csv",
    ]
    return all(p.is_file() for p in requeridos)


PROJECT_ROOT = REPO_DIR / "Actividad6"

# Intentar clonar solo si el repositorio aún no está en el entorno.
if not es_proyecto_actividad6(PROJECT_ROOT) and not (REPO_DIR / ".git").exists():
    print("Clonando el repositorio público...")
    resultado_git = subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        text=True,
        capture_output=True,
    )
    if resultado_git.returncode != 0:
        print(resultado_git.stderr[-3000:])

PROJECT_ROOT = REPO_DIR / "Actividad6"

# Si la rama pública todavía no contiene todos los archivos, usar el ZIP preparado.
if not es_proyecto_actividad6(PROJECT_ROOT):
    from google.colab import files
    print("No se encontró una carpeta Actividad6 completa en el repositorio.")
    print("Carga CIMA_Actividad6_GitHub.zip para continuar.")
    cargados = files.upload()
    zips = [Path(nombre) for nombre in cargados if nombre.lower().endswith(".zip")]
    assert zips, "No se cargó ningún ZIP. Se necesita CIMA_Actividad6_GitHub.zip."

    carpeta_respaldo = Path("/content/actividad6_respaldo")
    carpeta_respaldo.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as zf:
        # Extraer únicamente después de validar que el contenido pertenece al proyecto esperado.
        nombres = zf.namelist()
        assert any(n.replace("\\", "/").startswith("Actividad6/") for n in nombres), (
            "El ZIP no contiene la carpeta Actividad6/ esperada."
        )
        zf.extractall(carpeta_respaldo)

    candidatos = [
        carpeta_respaldo / "Actividad6",
        *[p.parent.parent for p in carpeta_respaldo.rglob("evaluacion.py")
          if p.parent.name == "fuentes"],
    ]
    PROJECT_ROOT = next((p for p in candidatos if es_proyecto_actividad6(p)), None)
    assert PROJECT_ROOT is not None, (
        "El ZIP se extrajo, pero faltan archivos requeridos. Revisa su estructura."
    )

print("Proyecto A6:", PROJECT_ROOT)
print("Notebook ejecutado con Python:", sys.version.split()[0])
print("Script de evaluación:", PROJECT_ROOT / "fuentes" / "evaluacion.py")

## 1. Verificar e instalar dependencias solo si hacen falta

En la sesión de Colab se verificó Python 3.13.16, scikit-learn 1.6.1, joblib 1.6.0 y MLflow 3.17.1. Para evitar el bloqueo de resolución, aquí NO se instala el archivo completo `requirements.txt`. Primero se verifican los módulos que utiliza `evaluacion.py`; si ya están disponibles y scikit-learn es 1.6.1, no se ejecuta `pip`.

MLflow no es necesario para calcular las métricas de esta actividad; sus evidencias de la Actividad 5 están respaldadas. Si falta algún paquete, se instala únicamente ese paquete.

In [ ]:
import importlib
import importlib.metadata
import subprocess
import sys

# Paquetes realmente importados por fuentes/evaluacion.py.
# No instalamos mlflow ni resolvemos requirements.txt en bloque.
PAQUETES_MINIMOS = {
    "numpy": "numpy>=1.26",
    "pandas": "pandas>=2.1,<3",
    "scipy": "scipy>=1.11,<2",
    "matplotlib": "matplotlib>=3.8,<4",
    "joblib": "joblib>=1.4,<2",
}

faltantes = []
for modulo, especificacion in PAQUETES_MINIMOS.items():
    try:
        importlib.import_module(modulo)
        print(f"[OK] {modulo}")
    except ImportError:
        print(f"[FALTA] {modulo}")
        faltantes.append(especificacion)

# Importante: scikit-learn 1.6.1 coincide con la versión con que se guardaron los modelos A5.
try:
    version_sklearn = importlib.metadata.version("scikit-learn")
except importlib.metadata.PackageNotFoundError:
    version_sklearn = None
if version_sklearn != "1.6.1":
    print(f"[AJUSTAR] scikit-learn actual: {version_sklearn}; se necesita 1.6.1")
    faltantes.append("scikit-learn==1.6.1")
else:
    print("[OK] scikit-learn==1.6.1")

# Solo llama pip cuando hay una dependencia ausente o una versión incompatible.
if faltantes:
    faltantes = list(dict.fromkeys(faltantes))
    print("Instalando únicamente:", faltantes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
    print("Instalación terminada. Si Colab pide reiniciar, reinicia y ejecuta desde el inicio.")
else:
    print("Las dependencias ya son compatibles. No se ejecutó pip install.")

# Verificación final, equivalente a las celdas que se probaron por separado.
import numpy as np
import pandas as pd
import scipy
import matplotlib
import joblib
import sklearn

print("\nPython:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("joblib:", joblib.__version__)
print("scikit-learn:", sklearn.__version__)
assert sklearn.__version__ == "1.6.1", (
    "La sesión no está usando scikit-learn 1.6.1. Si la instalación acaba de ejecutarse, "
    "reinicia el entorno de Colab y vuelve a ejecutar desde el inicio."
)


## 2. Validar datos, particiones y modelos

Antes de calcular métricas, verificamos la presencia de los CSV separados de entrenamiento y prueba y cargamos ambos modelos congelados. La prueba de 26 registros se conserva intacta; los umbrales se estiman a partir de datos OOF del conjunto de entrenamiento.

In [ ]:
import pandas as pd
import joblib

DATA_DIR = PROJECT_ROOT / "datos" / "datos_limp"
MODEL_DIR = PROJECT_ROOT / "modelos"
TRAIN_PATH = DATA_DIR / "entrenamiento.csv"
TEST_PATH = DATA_DIR / "prueba.csv"

train = pd.read_csv(TRAIN_PATH, encoding="utf-8-sig", keep_default_na=False)
test = pd.read_csv(TEST_PATH, encoding="utf-8-sig", keep_default_na=False)

for nombre, df in [("entrenamiento", train), ("prueba", test)]:
    requeridas = {"id", "texto_normalizado", "clase_real"}
    faltan = requeridas - set(df.columns)
    assert not faltan, f"A {nombre} le faltan columnas: {faltan}"
    assert df["texto_normalizado"].astype(str).str.strip().ne("").all(), (
        f"Se encontraron textos vacíos en {nombre}."
    )

assert len(train) == 102, f"Se esperaban 102 registros de entrenamiento; hay {len(train)}."
assert len(test) == 26, f"Se esperaban 26 registros de prueba; hay {len(test)}."
assert not (set(train["texto_normalizado"]) & set(test["texto_normalizado"])), (
    "Hay textos compartidos entre entrenamiento y prueba."
)

model_paths = {
    "Regresion_Logistica": MODEL_DIR / "Regresion_Logistica.joblib",
    "LinearSVC": MODEL_DIR / "LinearSVC.joblib",
}
models = {}
for nombre, ruta in model_paths.items():
    assert ruta.is_file(), f"No se encontró el modelo: {ruta}"
    models[nombre] = joblib.load(ruta)
    print(f"{nombre}: {type(models[nombre]).__name__}; clases={list(models[nombre].classes_)}")

print("\nTamaño de entrenamiento:", len(train))
print("Tamaño de prueba reservada:", len(test))
print("\nDistribución de clases (entrenamiento):")
display(train["clase_real"].value_counts().rename_axis("clase").to_frame("n"))
print("Distribución de clases (prueba):")
display(test["clase_real"].value_counts().rename_axis("clase").to_frame("n"))

## 3. Ejecutar el proceso reproducible de evaluación

El script `fuentes/evaluacion.py` realiza la evaluación de los modelos de A5, construye un baseline, calcula métricas por clase y ROC/AUC, aplica validación cruzada estratificada de cuatro pliegues con la configuración ya fijada, selecciona el umbral con puntuaciones OOF y genera la comparación estadística y la simulación retrospectiva de políticas.

**Importante:** no se realiza una nueva búsqueda de hiperparámetros y no se utiliza la prueba reservada para seleccionar el umbral.

In [ ]:
import subprocess
import sys

script_path = PROJECT_ROOT / "fuentes" / "evaluacion.py"
assert script_path.is_file(), f"No se encontró {script_path}"

# Capturar salida para conservar el diagnóstico si la evaluación falla.
proceso = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print(proceso.stdout)
if proceso.stderr.strip():
    print("\nMensajes del proceso:")
    print(proceso.stderr[-5000:])

if proceso.returncode != 0:
    raise RuntimeError(
        f"La evaluación terminó con código {proceso.returncode}. "
        "Revisa los mensajes anteriores antes de continuar."
    )

print("\nEVALUACIÓN TERMINADA CORRECTAMENTE.")

## 4. Consultar las métricas, la validación cruzada y los umbrales

Esta sección lee los CSV recién generados por el script. Así evitamos duplicar lógica analítica dentro del notebook y mantenemos una sola implementación de las métricas.

In [ ]:
from IPython.display import display

OUT = PROJECT_ROOT / "resultados"


def mostrar_resultado_csv(nombre: str, columnas=None):
    """Carga y presenta una tabla de resultados con validación de existencia."""
    ruta = OUT / nombre
    assert ruta.is_file(), f"No se generó el archivo esperado: {ruta}"
    tabla = pd.read_csv(ruta, encoding="utf-8-sig")
    print(f"\n{nombre} — {len(tabla)} filas")
    if columnas:
        disponibles = [c for c in columnas if c in tabla.columns]
        display(tabla[disponibles])
    else:
        display(tabla)
    return tabla


metricas = mostrar_resultado_csv(
    "metricas_actividad6.csv",
    ["modelo", "tipo", "accuracy", "precision_macro", "recall_macro",
     "f1_macro", "roc_auc_individual_vs_resto", "threshold",
     "false_eligibility_count", "manual_review_count"]
)
validacion_cv = mostrar_resultado_csv("validacion_cruzada.csv")
analisis_umbrales = mostrar_resultado_csv("analisis_umbrales.csv")
metricas_clase = mostrar_resultado_csv("metricas_por_clase.csv")
comparacion_estadistica = mostrar_resultado_csv("comparacion_estadistica.csv")
simulacion_ab = mostrar_resultado_csv("simulacion_ab_politicas.csv")

## 5. Revisar las visualizaciones generadas

Las figuras se producen desde el script y se guardan en `resultados/graficas/`. Se muestran aquí para inspección antes de versionarlas en GitHub y utilizarlas en el reporte.

In [ ]:
from IPython.display import display, Image

FIG_DIR = OUT / "graficas"
figuras = sorted(FIG_DIR.glob("*.png"))
assert figuras, f"No se encontraron gráficas en {FIG_DIR}"

print("Figuras generadas:", len(figuras))
for figura in figuras:
    print(figura.name)
    display(Image(filename=str(figura), width=820))

## 6. Validar los archivos de salida

La lista de control confirma que el script generó las tablas, los reportes y las visualizaciones esperadas. Si algún archivo falta, no se debe cerrar la entrega hasta revisar el error de la ejecución.

In [ ]:
archivos_esperados = [
    OUT / "metricas_actividad6.csv",
    OUT / "validacion_cruzada.csv",
    OUT / "analisis_umbrales.csv",
    OUT / "predicciones_actividad6.csv",
    OUT / "metricas_por_clase.csv",
    OUT / "comparacion_estadistica.csv",
    OUT / "simulacion_ab_politicas.csv",
    OUT / "metadatos_actividad6.json",
    OUT / "reportes" / "reportes_por_modelo.json",
]

# Cada modelo debe contar con su matriz de confusión y curva ROC.
for nombre in ["Regresion_Logistica", "LinearSVC"]:
    archivos_esperados.extend([
        FIG_DIR / f"matriz_{nombre}.png",
        FIG_DIR / f"roc_ovr_{nombre}.png",
    ])
archivos_esperados.extend([
    FIG_DIR / "roc_operativa_comparativa.png",
    FIG_DIR / "comparacion_cv_f1_macro.png",
])

faltantes = []
for ruta in archivos_esperados:
    ok = ruta.is_file()
    print(f"[{ 'OK' if ok else 'FALTA' }] {ruta.relative_to(PROJECT_ROOT)}")
    if not ok:
        faltantes.append(ruta)

print(f"\nArchivos esperados: {len(archivos_esperados)}")
print(f"Archivos faltantes: {len(faltantes)}")
if faltantes:
    raise FileNotFoundError("Faltan salidas requeridas. Revisa la ejecución del script.")
print("VALIDACIÓN DE SALIDAS SUPERADA.")

## 7. Exportar los resultados de esta ejecución

Esta celda genera un ZIP **solo con los resultados calculados** para descargarlos manualmente y revisarlos antes de copiarlos a GitHub. No es todavía el ZIP académico final, porque ese paquete también incorporará el reporte técnico y las evidencias del tablero Trello.

In [ ]:
import shutil
from google.colab import files

ZIP_RESULTADOS = Path("/content/CIMA_Actividad6_resultados.zip")
shutil.make_archive(str(ZIP_RESULTADOS.with_suffix("")), "zip", OUT)

assert ZIP_RESULTADOS.is_file() and ZIP_RESULTADOS.stat().st_size > 0
print("ZIP de resultados generado:", ZIP_RESULTADOS)
print("Tamaño (bytes):", ZIP_RESULTADOS.stat().st_size)
files.download(str(ZIP_RESULTADOS))

## 8. Interpretación, límites y cierre

- El baseline permite evaluar si los modelos aportan valor frente a una estrategia trivial; no constituye por sí solo una alternativa operativa recomendable.
- Regresión Logística produce probabilidades estimadas; LinearSVC produce márgenes de decisión que no deben describirse como probabilidades calibradas.
- El umbral se estima usando predicciones OOF del conjunto de entrenamiento. La prueba reservada se usa únicamente para evaluar la política seleccionada.
- Una falsa elegibilidad es una clasificación de una solicitud no individual como `INDIVIDUAL`; no significa que el sistema haya validado o autorizado un pago.
- La prueba exacta de McNemar compara errores emparejados. Un resultado no significativo no demuestra equivalencia, especialmente con 26 casos.
- La comparación A/B es una simulación retrospectiva y pareada sobre datos sintéticos. No es una prueba aleatorizada con usuarios y no demuestra causalidad ni ahorro real de tiempo.
- Con solo dos casos `AMBIGUA` en la prueba, la estimación de seguridad de esa categoría es muy inestable. Se requieren datos nuevos e independientes antes de considerar producción.

### Lista de cierre para GitHub

- [ ] Revisar las tablas y figuras generadas, sin sobrescribir los resultados históricos de la Actividad 5.
- [ ] Copiar los resultados validados a `Actividad6/resultados/` del repositorio.
- [ ] Actualizar `README.md` y `CHANGELOG.md` con la fecha, las versiones y los archivos generados.
- [ ] Incorporar el reporte técnico y las evidencias del tablero Trello.
- [ ] Preparar el ZIP académico final usando la misma versión de los archivos publicados en GitHub.